In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
print("running")

In [ ]:
# Configuration
DATASET = "shawon10/ckplus"
IMG = (48, 48)
BATCH = 32
EPOCHS = 40
SEED = 42
keras.utils.set_random_seed(SEED)

In [ ]:
# Download dataset
import kagglehub
DATA_DIR = pathlib.Path(kagglehub.dataset_download(DATASET))

In [ ]:
# Find emotion folders
matches = [p.parent for p in DATA_DIR.rglob("surprise") if p.is_dir()]
root = matches[0] if matches else DATA_DIR
folders = sorted([p for p in root.iterdir() if p.is_dir()])
class_names = [p.name for p in folders]
paths, labels = [], []
for i, folder in enumerate(folders):
    files = list(folder.glob("*.png")) + list(folder.glob("*.jpg"))
    paths.extend([str(p) for p in files])
    labels.extend([i] * len(files))
paths, labels = np.array(paths), np.array(labels)
num_classes = len(class_names)
print("Classes:", class_names)
print("Total images:", len(paths))

In [ ]:
# Split dataset
train_idx, temp_idx = train_test_split(
    np.arange(len(paths)), test_size=0.30,
    stratify=labels, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=labels[temp_idx], random_state=SEED
)

In [ ]:
# Load images
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.resize(img, IMG)
    return img, label
def make_dataset(indices, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (paths[indices], labels[indices])
    )
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(500, seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds = make_dataset(train_idx, True)
val_ds = make_dataset(val_idx)
test_ds = make_dataset(test_idx)

In [ ]:
# Class weights
weights = compute_class_weight(
    "balanced", classes=np.arange(num_classes),
    y=labels[train_idx]
)
class_weights = dict(enumerate(weights))
# Display sample images
plt.figure(figsize=(9, 3))
for images, lab in train_ds.take(1):
    for i in range(min(7, len(images))):
        plt.subplot(1, 7, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(class_names[int(lab[i])], fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Build CNN
model = keras.Sequential([
    layers.Input(shape=(48, 48, 1)),
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# Train model
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "best_ckplus_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
# Evaluate
loss, accuracy = model.evaluate(test_ds, verbose=0)
print("\nTest Accuracy:", round(accuracy * 100, 2), "%")
print("Test Loss:", round(loss, 4))

In [ ]:
# Accuracy and loss graphs
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
print("\nClassification Report:\n")
print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    zero_division=0
))
print("Macro F1:", round(f1_score(y_true, y_pred, average="macro"), 4))

In [ ]:
# Confusion matrix
sns.heatmap(
    confusion_matrix(y_true, y_pred),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Display predictions
for images, lab in test_ds.take(1):
    probs = model.predict(images[:8], verbose=0)
    plt.figure(figsize=(12, 4))
    for i in range(min(8, len(images))):
        p = np.argmax(probs[i])
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(
            f"Pred: {class_names[p]}\nActual: {class_names[int(lab[i])]}",
            color="green" if p == int(lab[i]) else "red",
            fontsize=8
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Save model
model.save("ckplus_cnn_final.keras")
print("\nModel saved successfully!")
print("CNN Facial Emotion Recognition Completed!")